MD**Intelligent Vision Systems · Lab 5 · Self-Driving Car Vision: Finding Lane Lines**  
Companion to **Lecture 5 (Part II) — Computer Vision for Self-Driving Cars**  
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_5/06-bug-hunt.ipynb) &nbsp; [Lecture 5 Part II slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/05_SDC_Lane_Finding.html#/common-pitfalls) &nbsp; [All Lab 5 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-5-lane-lines)

# Lab 5F · Bug Hunt 🐞

_Lecture 5 Part II, slide “Common pitfalls”._ **Work in pairs: one types (driver), one reads and thinks (navigator); swap at every bug.**

A teammate wrote this lane-finding code and it is full of the classic mistakes. There are **8 planted bugs**, one per
section. For each one:

1. **Predict** what the code will do before you run it (write it down!).
2. Run the code and the ✅ check. If it prints 💡, find and fix the bug **in the code cell**, then re-run both cells.
3. Only open the hint when you are stuck.

The scoreboard at the end shows how many bugs your pair has fixed. First pair to 8/8 shows the TA! 🏆

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab images if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_5"
for f in ['lab5_checks.py', 'test.jpg', 'test_images/solidYellowCurve2.jpg']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("data ready")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2
import lab5_checks as C

# Bug 3 needs a PNG version of test.jpg
cv2.imwrite("test.png", cv2.imread("test.jpg"))
print("ready - start hunting!")

MD## Bug 1 · Where did the yellow line go?

`yellow_mask()` should find the yellow lane line using HLS, but it finds (almost) nothing.

<details><summary>💡 Hint</summary>

What channel order does `cv2.imread` return? Which conversion code matches it?
</details>

In [ ]:
def yellow_mask(path):
    img = cv2.imread(path)                              # load the image
    hls = cv2.cvtColor(img, cv2.COLOR_RGB2HLS)          # to HLS
    return cv2.inRange(hls, (15, 30, 115), (35, 204, 255))

mask = yellow_mask("test_images/solidYellowCurve2.jpg")
plt.imshow(mask, cmap="gray"); plt.title("yellow mask"); plt.axis("off"); plt.show()

In [ ]:
# ✅ Check bug 1
C.check_bug(1, yellow_mask)

MD## Bug 2 · The disappearing original

We black out dark pixels to keep the bright lane lines… but afterwards the original image is ruined too.

<details><summary>💡 Hint</summary>

Does `out = image` make a copy in NumPy? Compare with `np.copy`.
</details>

In [ ]:
def select_bright(image, t=200):
    out = image                                          # work on a copy of the image
    out[(image < t).any(axis=2)] = 0
    return out

original = mpimg.imread("test.jpg").copy()
selected = select_bright(original)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].imshow(original); ax[0].set_title("original (should be untouched)")
ax[1].imshow(selected); ax[1].set_title("selected")
for a in ax: a.axis("off")
plt.show()

In [ ]:
# ✅ Check bug 2
C.check_bug(2, original, selected)

MD## Bug 3 · Nothing survives the threshold

The same selection on `test.png` keeps zero pixels, although the image is identical to `test.jpg`.

<details><summary>💡 Hint</summary>

Print `img.dtype` and `img.max()`. What range do PNGs read with `mpimg` use?
</details>

In [ ]:
img = mpimg.imread("test.png")
kept = (img >= 200).all(axis=2)
kept_fraction = kept.mean()
print("dtype:", img.dtype, "| max value:", img.max(), "| kept:", kept_fraction)

In [ ]:
# ✅ Check bug 3
C.check_bug(3, kept_fraction)

MD## Bug 4 · The blur that crashes

Smoothing before Canny should be easy…

<details><summary>💡 Hint</summary>

Read the error message. What must be true about a Gaussian kernel size?
</details>

In [ ]:
def blur(gray):
    kernel_size = 4
    return cv2.GaussianBlur(gray, (kernel_size, kernel_size), 0)

out = blur(cv2.cvtColor(mpimg.imread("test.jpg"), cv2.COLOR_RGB2GRAY))
print(out.shape)

In [ ]:
# ✅ Check bug 4
C.check_bug(4, blur)

MD## Bug 5 · The region mask won't draw

A trapezoid region of interest, written with fractions of the image size.

<details><summary>💡 Hint</summary>

What data type does `cv2.fillPoly` want for its points? Check `vertices.dtype`.
</details>

In [ ]:
def region_mask(shape):
    h, w = shape[:2]
    vertices = np.array([[(0.05 * w, h), (0.46 * w, 0.6 * h), (0.54 * w, 0.6 * h), (0.95 * w, h)]])
    mask = np.zeros((h, w), np.uint8)
    cv2.fillPoly(mask, vertices, 255)
    return mask

plt.imshow(region_mask((540, 960)), cmap="gray"); plt.axis("off"); plt.show()

In [ ]:
# ✅ Check bug 5
C.check_bug(5, region_mask)

MD## Bug 6 · It crashed on a dark frame

Works on sunny frames, crashes on a frame with no lines (e.g. inside a tunnel).

<details><summary>💡 Hint</summary>

What does `cv2.HoughLinesP` return when it finds nothing? Print it.
</details>

In [ ]:
def hough_segments(edges):
    lines = cv2.HoughLinesP(edges, 2, np.pi / 180, 15, minLineLength=40, maxLineGap=20)
    return [tuple(l[0]) for l in lines]

print(hough_segments(np.zeros((540, 960), np.uint8)))

In [ ]:
# ✅ Check bug 6
C.check_bug(6, hough_segments)

MD## Bug 7 · Left is right and right is left

Segments are split into left and right lanes by the sign of their slope.

<details><summary>💡 Hint</summary>

Draw the left lane on paper with y growing DOWN the page. As x increases, does y go up or down?
</details>

In [ ]:
def split_left_right(segments):
    left, right = [], []
    for x1, y1, x2, y2 in segments:
        m = (y2 - y1) / (x2 - x1)
        if m > 0:
            left.append((x1, y1, x2, y2))
        else:
            right.append((x1, y1, x2, y2))
    return left, right

print(split_left_right([(200, 500, 400, 350), (760, 500, 560, 350)]))

In [ ]:
# ✅ Check bug 7
C.check_bug(7, split_left_right)

MD## Bug 8 · Division by zero

Edge maps sometimes contain perfectly vertical segments (poles, car edges).

<details><summary>💡 Hint</summary>

What is `x2 - x1` for a vertical segment? Skip those before dividing.
</details>

In [ ]:
def lane_slopes(segments):
    slopes = []
    for x1, y1, x2, y2 in segments:
        m = (y2 - y1) / (x2 - x1)
        if abs(m) > 0.4:
            slopes.append(m)
    return slopes

print(lane_slopes([(200, 500, 400, 350), (300, 100, 300, 400), (760, 500, 560, 350)]))

In [ ]:
# ✅ Check bug 8
C.check_bug(8, lane_slopes)

MD## 🏆 Scoreboard

In [ ]:
C.bug_hunt_summary()

MD## Reflect (2 minutes, with your partner)

1. Which bug took you longest, and what finally gave it away (error message, a print, a plot)?
2. Which of these bugs would **not** crash but silently give wrong lanes in a real car? Why are those the dangerous ones?
3. Write one **assert** you would add to your Assignment 2 pipeline to catch one of these bugs early.